# **3일차 팀 프로젝트: 테이블 데이터 조회 시스템 구축**

## 프로젝트 목표
1. 팀에서 선정한 CSV 테이블 데이터를 Supabase에 적재
2. SQL 쿼리로 데이터 조회 테스트
3. Text2SQL 시스템 구현 및 테스트

## 구현 단계
- 환경 설정 확인
- CSV 파일 확인 및 탐색
- Supabase 연결
- CSV 데이터 업로드
- SQL 쿼리 테스트
- Text2SQL 시스템 구현 및 테스트

## 0. 환경 변수 설정

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# OpenAI API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Supabase 설정 확인
if os.environ.get("SUPABASE_DB_URL"):
    print("✓ Supabase DB URL이 설정되었습니다.")
else:
    print("✗ Supabase DB URL이 필요합니다.")
    print("  .env 파일에 SUPABASE_DB_URL을 추가하세요.")

✓ OpenAI API Key가 설정되었습니다.
✓ Supabase DB URL이 설정되었습니다.


## 1. CSV 파일 확인 및 탐색

전국 시도별 인구와 시·군·구 GRDP CSV 파일을 불러와 구조와 품질을 확인합니다.

**중요:** 여러 CSV 파일을 사용하는 경우, 테이블 간 관계(Foreign Key)를 고려하여 업로드 순서를 결정하세요.
- 부모 테이블 → 자식 테이블 순서로 업로드

In [2]:
import pandas as pd

# TODO: 팀의 CSV 파일 경로를 입력하세요
# 여러 파일이 있다면 dict 형태로 구성
# 예시:
# csv_files = {
#     "table1": "../datasets/your_table1.csv",
#     "table2": "../datasets/your_table2.csv"
# }

POPULATION_TABLE = "population"
GRDP_TABLE = "GRDP"

csv_files = {
    POPULATION_TABLE: "../datasets/인구총조사_인구_시도_시_군_구__20260820152244.csv",
    GRDP_TABLE: "../datasets/GRDP_시_군_구__20260820151917.csv",
    # 필요한 만큼 추가
}

# CSV 파일 로드 및 확인
dataframes = {}

for table_name, file_path in csv_files.items():
    try:
        if table_name == GRDP_TABLE:
            df = pd.read_csv(file_path, encoding="utf-8", header=1)
            df.columns = [
                "시도", "시군구",
                "2018_당해년가격", "2018_기준년가격",
                "2019_당해년가격", "2019_기준년가격",
                "2020_당해년가격", "2020_기준년가격",
                "2021_당해년가격", "2021_기준년가격",
                "2022_당해년가격", "2022_기준년가격",
            ]
        else:
            df = pd.read_csv(file_path, encoding="utf-8")
        dataframes[table_name] = df

        print("=" * 80)
        print(f"📋 {table_name} 테이블")
        print("=" * 80)
        print(f"\n행 수: {len(df)}")
        print(f"컬럼: {list(df.columns)}")
        print(f"\n첫 5개 행:")
        print(df.head())
        print(f"\n데이터 타입:")
        print(df.dtypes)
        print("\n")

    except Exception as e:
        print(f"✗ {table_name} 로드 실패: {e}\n")

print(f"\n✓ 총 {len(dataframes)}개의 테이블 로드 완료")

📋 population 테이블

행 수: 18
컬럼: ['행정구역별(읍면동)', '연령별', '2021', '2022', '2023', '2024', '2025']

첫 5개 행:
  행정구역별(읍면동) 연령별      2021      2022      2023      2024      2025
0         전국  합계  51738071  51692272  51774521  51805547  51817499
1      서울특별시  합계   9472127   9417469   9384512   9335444   9315066
2      부산광역시  합계   3324335   3295760   3279604   3257256   3235362
3      대구광역시  합계   2387911   2366852   2379188   2369335   2363643
4      인천광역시  합계   2957044   2989125   3025950   3058033   3093590

데이터 타입:
행정구역별(읍면동)      str
연령별             str
2021          int64
2022          int64
2023          int64
2024          int64
2025          int64
dtype: object


✗ GRDP 로드 실패: [Errno 2] No such file or directory: '../datasets/GRDP_시_군_구__20260820151917.csv'


✓ 총 1개의 테이블 로드 완료


## 2. 데이터 탐색 및 통계

2021~2025 시도별 인구와 2018~2022 지역별 GRDP의 범위, 결측값, 순위와 증감률을 탐색합니다.

In [3]:
# TODO: 각 테이블의 주요 통계를 확인하세요
# 예시:
# - 특정 컬럼의 고유값 개수
# - 카테고리별 데이터 분포
# - 결측치 확인

for table_name, df in dataframes.items():
    print(f"\n{'='*80}")
    print(f"📊 {table_name} 통계")
    print("="*80)

    # 기본 통계
    print("\n[기본 정보]")
    print(df.info())

    # 결측치 확인
    print("\n[결측치]")
    null_counts = df.isnull().sum()
    if null_counts.sum() > 0:
        print(null_counts[null_counts > 0])
    else:
        print("결측치 없음")

    # TODO: 팀 데이터에 맞는 추가 탐색 코드를 작성하세요
    # 예시:
    # print("\n[카테고리 분포]")
    # print(df['YOUR_COLUMN'].value_counts())
    if table_name == POPULATION_TABLE:
        population_regions = df[df["행정구역별(읍면동)"] != "전국"].copy()
        population_regions["2021~2022 인구증감률(%)"] = ((population_regions["2022"] - population_regions["2021"]) / population_regions["2021"] * 100).round(2)
        print("\n[2022년 시도별 인구 상위 5개]")
        print(population_regions.nlargest(5, "2022")[["행정구역별(읍면동)", "2021", "2022", "2021~2022 인구증감률(%)"]])
    elif table_name == GRDP_TABLE:
        grdp_subtotal = df[(df["시군구"] == "소계") & (df["2022_당해년가격"] != "-")].copy()
        for column in ["2021_당해년가격", "2021_기준년가격", "2022_당해년가격", "2022_기준년가격"]:
            grdp_subtotal[column] = pd.to_numeric(grdp_subtotal[column])
        grdp_subtotal["2021~2022 실질GRDP증감률(%)"] = ((grdp_subtotal["2022_기준년가격"] - grdp_subtotal["2021_기준년가격"]) / grdp_subtotal["2021_기준년가격"] * 100).round(2)
        print("\n[2022년 시도별 명목 GRDP 상위 5개]")
        print(grdp_subtotal.nlargest(5, "2022_당해년가격")[["시도", "2022_당해년가격", "2022_기준년가격", "2021~2022 실질GRDP증감률(%)"]])


📊 population 통계

[기본 정보]
<class 'pandas.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   행정구역별(읍면동)  18 non-null     str  
 1   연령별         18 non-null     str  
 2   2021        18 non-null     int64
 3   2022        18 non-null     int64
 4   2023        18 non-null     int64
 5   2024        18 non-null     int64
 6   2025        18 non-null     int64
dtypes: int64(5), str(2)
memory usage: 1.5 KB
None

[결측치]
결측치 없음

[2022년 시도별 인구 상위 5개]
   행정구역별(읍면동)      2021      2022  2021~2022 인구증감률(%)
9         경기도  13652529  13717827                0.48
1       서울특별시   9472127   9417469               -0.58
2       부산광역시   3324335   3295760               -0.86
16       경상남도   3305931   3280829               -0.76
4       인천광역시   2957044   2989125                1.08


## 3. Supabase PostgreSQL 연결

In [4]:
from langchain_community.utilities import SQLDatabase

supabase_db_url = os.getenv("SUPABASE_DB_URL")

if not supabase_db_url:
    raise Exception("SUPABASE_DB_URL이 설정되지 않았습니다. .env 파일을 확인하세요.")

print("Supabase PostgreSQL 연결 중...\n")

try:
    # LangChain SQLDatabase로 PostgreSQL 연결
    db = SQLDatabase.from_uri(supabase_db_url)

    print("✓ PostgreSQL 연결 성공!\n")
    print("현재 테이블 목록:")
    tables = db.get_usable_table_names()
    print(tables)

    required_tables = {POPULATION_TABLE, GRDP_TABLE}
    missing_tables = required_tables - set(tables)
    if missing_tables:
        raise Exception(f"Supabase에 필요한 테이블이 없습니다: {sorted(missing_tables)}")

except Exception as e:
    print(f"✗ 연결 실패: {e}")
    raise

C:\Users\user\AppData\Local\Temp\ipykernel_26928\2332592179.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


Supabase PostgreSQL 연결 중...

✓ PostgreSQL 연결 성공!

현재 테이블 목록:
['GRDP', 'aviation_accidents', 'departments', 'office_floors', 'organizations', 'population', 'railway_accidents', '성남시_인구_데이터', '화성시_인구_데이터']


## 4. Supabase에 CSV 데이터 업로드

### Supabase 대시보드 (GUI)
1. Supabase 대시보드 접속
2. Table Editor → Import data from CSV
3. 인구 CSV는 `population`, GRDP CSV는 `GRDP` 테이블명으로 업로드
4. GRDP의 `2018`~`2022_1` 컬럼은 당해년가격과 2020년 기준년가격이 교대로 배치됨
5. GRDP 첫 데이터 행에는 실제 지표명이 들어 있으므로 분석 쿼리에서 제외

## 5. 업로드 확인 및 스키마 탐색

In [5]:
# 데이터베이스 다시 연결 (업로드 후 스키마 갱신)
db = SQLDatabase.from_uri(supabase_db_url)

print("=== 데이터베이스 스키마 ===")
print(db.table_info)

print("\n" + "="*80 + "\n")

# 각 테이블의 샘플 데이터
for table in [POPULATION_TABLE, GRDP_TABLE]:
    print(f"{table} 테이블 샘플:")
    try:
        result = db.run(f'SELECT * FROM "{table}" LIMIT 3')
        print(result)
    except Exception as e:
        print(f"조회 실패: {e}")
    print()

=== 데이터베이스 스키마 ===

CREATE TABLE "GRDP" (
	"행정구역별(1)" TEXT, 
	"행정구역별(2)" TEXT, 
	"2018" TEXT, 
	"2018_1" TEXT, 
	"2019" TEXT, 
	"2019_1" TEXT, 
	"2020" TEXT, 
	"2020_1" TEXT, 
	"2021" TEXT, 
	"2021_1" TEXT, 
	"2022" TEXT, 
	"2022_1" TEXT
)

/*
3 rows from GRDP table:
행정구역별(1)	행정구역별(2)	2018	2018_1	2019	2019_1	2020	2020_1	2021	2021_1	2022	2022_1
행정구역별(1)	행정구역별(2)	당해년가격	2020년 기준년가격	당해년가격	2020년 기준년가격	당해년가격	2020년 기준년가격	당해년가격	2020년 기준년가격	당해년가격	2020년 기준년가격
서울특별시	소계	451385832	460496443	465940680	475206455	472863219	472863219	507460707	491388714	528484481	504359507
서울특별시	종로구	30763051	31200879	31952991	32541781	31074021	31086396	33155488	31614190	34298656	32080851
*/


CREATE TABLE "성남시_인구_데이터" (
	"구별" TEXT, 
	"동" TEXT, 
	"인구수_계" BIGINT, 
	"인구수_남" BIGINT, 
	"인구수_여" BIGINT, 
	"19세 이상_계" BIGINT, 
	"19세 이상_남" BIGINT, 
	"19세 이상_여" BIGINT, 
	"65세 이상_계" BIGINT, 
	"65세 이상_남자" BIGINT, 
	"65세 이상_여자" BIGINT, 
	"세대수" BIGINT, 
	"재외국민" BIGINT, 
	"데이터기준일자" TEXT
)

/*
3 rows from 성남시_인구_데이터 table:
구별	동	인구수_계	인

## 6. SQL 쿼리 테스트

2022년 GRDP 순위, 인구–GRDP JOIN과 1인당 GRDP, 공통지역 GRDP 합계와 증감률 쿼리를 테스트합니다.

In [6]:
# TODO: 기본 조회 쿼리 작성
# 예시: 특정 조건으로 데이터 조회

query = f"""
SELECT
    "행정구역별(1)" AS "지역",
    "2022"::numeric AS "2022년 명목 GRDP",
    "2022_1"::numeric AS "2022년 실질 GRDP",
    ROUND((("2022"::numeric - "2021"::numeric) / NULLIF("2021"::numeric, 0) * 100)::numeric, 2) AS "명목 GRDP 증감률(%)",
    ROUND((("2022_1"::numeric - "2021_1"::numeric) / NULLIF("2021_1"::numeric, 0) * 100)::numeric, 2) AS "실질 GRDP 증감률(%)"
FROM "{GRDP_TABLE}"
WHERE "행정구역별(2)" = '소계'
  AND "2022" <> '-'
ORDER BY "2022년 명목 GRDP" DESC;
"""

print("실행 쿼리:")
print(query)
print("\n결과:")

try:
    result = db.run(query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리:

SELECT
    "행정구역별(1)" AS "지역",
    "2022"::numeric AS "2022년 명목 GRDP",
    "2022_1"::numeric AS "2022년 실질 GRDP",
    ROUND((("2022"::numeric - "2021"::numeric) / NULLIF("2021"::numeric, 0) * 100)::numeric, 2) AS "명목 GRDP 증감률(%)",
    ROUND((("2022_1"::numeric - "2021_1"::numeric) / NULLIF("2021_1"::numeric, 0) * 100)::numeric, 2) AS "실질 GRDP 증감률(%)"
FROM "GRDP"
WHERE "행정구역별(2)" = '소계'
  AND "2022" <> '-'
ORDER BY "2022년 명목 GRDP" DESC;


결과:
[('경기도', Decimal('587328570'), Decimal('571942091'), Decimal('4.25'), Decimal('3.24')), ('서울특별시', Decimal('528484481'), Decimal('504359507'), Decimal('4.14'), Decimal('2.64')), ('충청남도', Decimal('140928250'), Decimal('133543615'), Decimal('3.71'), Decimal('2.84')), ('경상남도', Decimal('126917220'), Decimal('121090992'), Decimal('7.35'), Decimal('3.53')), ('경상북도', Decimal('121411361'), Decimal('115612472'), Decimal('2.27'), Decimal('0.63')), ('부산광역시', Decimal('113844056'), Decimal('102497863'), Decimal('6.89'), Decimal('1.73')), ('인천광역시', Decimal

In [7]:
# TODO: JOIN 쿼리 작성 (여러 테이블을 사용하는 경우)
# 예시: 두 테이블을 조인하여 데이터 조회

join_query = f"""
WITH population_2022 AS (
    SELECT
        CASE
            WHEN "행정구역별(읍면동)" IN ('광주광역시', '전라남도') THEN '전남광주통합특별시'
            ELSE "행정구역별(읍면동)"
        END AS "지역",
        SUM("2021"::numeric) AS "2021년 인구",
        SUM("2022"::numeric) AS "2022년 인구"
    FROM "{POPULATION_TABLE}"
    WHERE "행정구역별(읍면동)" <> '전국'
    GROUP BY "지역"
),
grdp_2022 AS (
    SELECT
        "행정구역별(1)" AS "지역",
        "2021_1"::numeric AS "2021년 실질 GRDP",
        "2022"::numeric AS "2022년 명목 GRDP",
        "2022_1"::numeric AS "2022년 실질 GRDP"
    FROM "{GRDP_TABLE}"
    WHERE "행정구역별(2)" = '소계'
      AND "2022" <> '-'
)
SELECT
    p."지역",
    p."2022년 인구",
    g."2022년 명목 GRDP",
    g."2022년 실질 GRDP",
    ROUND((g."2022년 명목 GRDP" / NULLIF(p."2022년 인구", 0))::numeric, 2) AS "1인당 명목 GRDP(백만원)",
    ROUND(((p."2022년 인구" - p."2021년 인구") / NULLIF(p."2021년 인구", 0) * 100)::numeric, 2) AS "인구 증감률(%)",
    ROUND(((g."2022년 실질 GRDP" - g."2021년 실질 GRDP") / NULLIF(g."2021년 실질 GRDP", 0) * 100)::numeric, 2) AS "실질 GRDP 증감률(%)"
FROM population_2022 AS p
INNER JOIN grdp_2022 AS g ON p."지역" = g."지역"
ORDER BY "1인당 명목 GRDP(백만원)" DESC;
"""

print("실행 쿼리:")
print(join_query)
print("\n결과:")

try:
    result = db.run(join_query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리:

WITH population_2022 AS (
    SELECT
        CASE
            WHEN "행정구역별(읍면동)" IN ('광주광역시', '전라남도') THEN '전남광주통합특별시'
            ELSE "행정구역별(읍면동)"
        END AS "지역",
        SUM("2021"::numeric) AS "2021년 인구",
        SUM("2022"::numeric) AS "2022년 인구"
    FROM "population"
    WHERE "행정구역별(읍면동)" <> '전국'
    GROUP BY "지역"
),
grdp_2022 AS (
    SELECT
        "행정구역별(1)" AS "지역",
        "2021_1"::numeric AS "2021년 실질 GRDP",
        "2022"::numeric AS "2022년 명목 GRDP",
        "2022_1"::numeric AS "2022년 실질 GRDP"
    FROM "GRDP"
    WHERE "행정구역별(2)" = '소계'
      AND "2022" <> '-'
)
SELECT
    p."지역",
    p."2022년 인구",
    g."2022년 명목 GRDP",
    g."2022년 실질 GRDP",
    ROUND((g."2022년 명목 GRDP" / NULLIF(p."2022년 인구", 0))::numeric, 2) AS "1인당 명목 GRDP(백만원)",
    ROUND(((p."2022년 인구" - p."2021년 인구") / NULLIF(p."2021년 인구", 0) * 100)::numeric, 2) AS "인구 증감률(%)",
    ROUND(((g."2022년 실질 GRDP" - g."2021년 실질 GRDP") / NULLIF(g."2021년 실질 GRDP", 0) * 100)::numeric, 2) AS "실질 GRDP 증감률(%)"
FRO

In [8]:
# TODO: 집계(Aggregation) 쿼리 작성
# 예시: GROUP BY, COUNT, SUM, AVG 등 사용

aggregation_query = f"""
SELECT
    SUM("2021"::numeric) AS "2021년 명목 GRDP 합계",
    SUM("2022"::numeric) AS "2022년 명목 GRDP 합계",
    SUM("2021_1"::numeric) AS "2021년 실질 GRDP 합계",
    SUM("2022_1"::numeric) AS "2022년 실질 GRDP 합계",
    ROUND(((SUM("2022"::numeric) - SUM("2021"::numeric)) / NULLIF(SUM("2021"::numeric), 0) * 100)::numeric, 2) AS "명목 GRDP 증감률(%)",
    ROUND(((SUM("2022_1"::numeric) - SUM("2021_1"::numeric)) / NULLIF(SUM("2021_1"::numeric), 0) * 100)::numeric, 2) AS "실질 GRDP 증감률(%)"
FROM "{GRDP_TABLE}"
WHERE "행정구역별(2)" = '소계'
  AND "2021" <> '-'
  AND "2022" <> '-';
"""

print("실행 쿼리:")
print(aggregation_query)
print("\n결과:")

try:
    result = db.run(aggregation_query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리:

SELECT
    SUM("2021"::numeric) AS "2021년 명목 GRDP 합계",
    SUM("2022"::numeric) AS "2022년 명목 GRDP 합계",
    SUM("2021_1"::numeric) AS "2021년 실질 GRDP 합계",
    SUM("2022_1"::numeric) AS "2022년 실질 GRDP 합계",
    ROUND(((SUM("2022"::numeric) - SUM("2021"::numeric)) / NULLIF(SUM("2021"::numeric), 0) * 100)::numeric, 2) AS "명목 GRDP 증감률(%)",
    ROUND(((SUM("2022_1"::numeric) - SUM("2021_1"::numeric)) / NULLIF(SUM("2021_1"::numeric), 0) * 100)::numeric, 2) AS "실질 GRDP 증감률(%)"
FROM "GRDP"
WHERE "행정구역별(2)" = '소계'
  AND "2021" <> '-'
  AND "2022" <> '-';


결과:
[(Decimal('2067645432'), Decimal('2166147945'), Decimal('2008572746'), Decimal('2067472105'), Decimal('4.76'), Decimal('2.93'))]


## 7. Text2SQL 함수 구현

인구와 GRDP의 연도·가격 기준·지역명 차이를 반영한 시스템 프롬프트를 사용합니다.

In [9]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

llm = init_chat_model("gpt-5.4-mini")

SQL_STOP_MARKERS = (
    chr(52280) + chr(44256) + ":",
    chr(49444) + chr(47749) + ":",
    chr(51452) + chr(51032) + ":",
)

def clean_sql_output(content: str) -> str:
    cleaned = content.replace("```sql", "").replace("```", "").strip()
    upper = cleaned.upper()
    starts = [position for position in [upper.find("SELECT"), upper.find("WITH")] if position >= 0]
    if not starts:
        raise ValueError("응답에서 SELECT 또는 WITH 쿼리를 찾을 수 없습니다.")
    cleaned = cleaned[min(starts):]
    sql_lines = []
    for line in cleaned.splitlines():
        if line.strip().startswith(SQL_STOP_MARKERS):
            break
        sql_lines.append(line)
    cleaned = "\n".join(sql_lines).strip()
    if ";" in cleaned:
        cleaned = cleaned.split(";", 1)[0] + ";"
    else:
        lines = []
        for line in cleaned.splitlines():
            if line.strip().startswith(SQL_STOP_MARKERS):
                break
            lines.append(line)
        cleaned = "\n".join(lines).strip()
    return cleaned

def text_to_sql(question: str, db: SQLDatabase) -> str:
    """
    자연어 질문을 SQL로 변환
    """
    # TODO: 시스템 프롬프트를 팀 데이터에 맞게 수정하세요
    system_prompt = f"""
    당신은 전국 인구와 지역내총생산(GRDP) 통계 전문 SQL 작성자입니다.
    사용자의 질문을 아래 PostgreSQL 데이터에 맞는 SQL 쿼리로 변환하세요.

    데이터베이스 스키마:
    {db.table_info}

    """

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=question)
    ]

    response = llm.invoke(messages)
    sql = clean_sql_output(response.content)

    # 코드 블록 제거
    if sql.startswith("```"):
        lines = sql.split("\n")
        sql = "\n".join(lines[1:-1]) if len(lines) > 2 else sql
        sql = sql.replace("sql", "").replace("```", "").strip()

    return sql

def fix_sql(question: str, sql: str, error: str, db: SQLDatabase) -> str:
    system_prompt = f"""
    당신은 PostgreSQL 오류를 수정하는 SQL 전문가입니다.
    아래 스키마와 오류를 확인하고 실행 가능한 SELECT 쿼리 하나만 반환하세요.

    데이터베이스 스키마:
    {db.table_info}
    """

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"질문: {question}\n\n오류가 발생한 SQL:\n{sql}\n\n오류:\n{error}")
    ]

    response = llm.invoke(messages)
    fixed_sql = clean_sql_output(response.content)

    if fixed_sql.startswith("```"):
        lines = fixed_sql.split("\n")
        fixed_sql = "\n".join(lines[1:-1]) if len(lines) > 2 else fixed_sql
        fixed_sql = fixed_sql.replace("sql", "").replace("```", "").strip()

    return fixed_sql

print("✓ Text2SQL 함수 준비 완료")

✓ Text2SQL 함수 준비 완료


## 8. Text2SQL 테스트

2022년 지역별 1인당 명목 GRDP를 묻는 자연어 질문으로 SQL 생성을 테스트합니다.

In [10]:
# TODO: 팀 데이터에 맞는 질문을 작성하세요
question = "2022년 지역별 1인당 명목 GRDP가 높은 5곳을 알려줘."

print(f"질문: {question}\n")
print("="*80)

# SQL 생성
sql = clean_sql_output(text_to_sql(question, db))
print(f"\n생성된 SQL:")
print(sql)
print()

# SQL 실행
print("="*80)
print("\n실행 결과:")
try:
    result = db.run(sql)
    print(result)
except Exception as e:
    print(f"실행 오류: {e}")
    print("SQL을 교정하여 다시 실행합니다.")
    sql = clean_sql_output(fix_sql(question, sql, str(e), db))
    print(f"\n교정된 SQL:\n{sql}\n")
    result = db.run(sql)
    print(result)

질문: 2022년 지역별 1인당 명목 GRDP가 높은 5곳을 알려줘.


생성된 SQL:
SELECT
    "행정구역별(1)" AS 지역,
    REPLACE("2022", ',', '')::numeric AS "2022년_명목GRDP",
    REPLACE("2022_1", ',', '')::numeric AS "2022년_2020기준년GRDP",
    ROUND(
        REPLACE("2022", ',', '')::numeric / NULLIF(
            CASE
                WHEN "행정구역별(2)" = '소계' THEN NULL
                ELSE 1
            END, 0
        )
    , 0) AS "1인당GRDP"
FROM "GRDP"
WHERE "행정구역별(2)" <> '소계'
  AND "행정구역별(1)" <> '행정구역별(1)'
ORDER BY "1인당GRDP" DESC
LIMIT 5;


실행 결과:
실행 오류: (psycopg2.errors.InvalidTextRepresentation) invalid input syntax for type numeric: "-"

[SQL: SELECT
    "행정구역별(1)" AS 지역,
    REPLACE("2022", ',', '')::numeric AS "2022년_명목GRDP",
    REPLACE("2022_1", ',', '')::numeric AS "2022년_2020기준년GRDP",
    ROUND(
        REPLACE("2022", ',', '')::numeric / NULLIF(
            CASE
                WHEN "행정구역별(2)" = '소계' THEN NULL
                ELSE 1
            END, 0
        )
    , 0) AS "1인당GRDP"
FROM "GRDP"
WHERE "행정구역별(2)" <> '

## 9. 완전한 Text2SQL 시스템 (SQL 실행 + 자연어 답변)

SQL 결과만 근거로 연도·가격 기준·지역 결합 범위를 설명하는 답변 프롬프트를 사용합니다.

In [11]:
def query_database(question: str, db: SQLDatabase) -> str:
    """
    자연어 질문 → SQL 생성 → 실행 → 자연어 답변
    """
    # 1. SQL 생성
    print(f"[1] SQL 생성 중...")
    sql = text_to_sql(question, db)
    print(f"    {sql}\n")

    # 2. SQL 실행
    print(f"[2] SQL 실행 중...")
    try:
        result = db.run(sql)
        print(f"    실행 완료\n")
    except Exception as e:
        print(f"    실행 오류: {e}")
        print(f"    SQL 교정 후 재실행 중...")
        try:
            sql = fix_sql(question, sql, str(e), db)
            print(f"    교정된 SQL: {sql}\n")
            result = db.run(sql)
            print(f"    재실행 완료\n")
        except Exception as retry_error:
            return f"SQL 실행 오류: {retry_error}"

    # 3. 자연어 답변 생성
    print(f"[3] 답변 생성 중...")

    # TODO: 시스템 프롬프트를 팀 데이터 도메인에 맞게 수정하세요
    system_prompt = """
    당신은 전국 인구와 지역내총생산(GRDP) 통계 데이터 분석 전문가입니다.
    SQL 쿼리 결과에 포함된 수치만 근거로 사용자의 질문에 답변하세요.
    인구와 GRDP를 결합할 때 사용한 공통연도, 명목·실질 가격 기준, 포함 지역을 밝혀주세요.
    답변은 핵심 수치와 기준을 포함하여 명확하고 이해하기 쉽게 작성하세요.
    """

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"""
        질문: {question}

        실행한 SQL:
        {sql}

        쿼리 결과:
        {result}

        위 결과를 바탕으로 질문에 답변해주세요.
        """)
    ]

    response = llm.invoke(messages)
    return response.content

print("✓ 완전한 Text2SQL 시스템 준비 완료")

✓ 완전한 Text2SQL 시스템 준비 완료


In [12]:
from IPython.display import Markdown, display

# TODO: 팀 데이터에 맞는 질문을 작성하세요
question = "2021년에서 2022년 사이 지역별 인구증감률과 실질 GRDP 증감률을 비교해줘."

print(f"질문: {question}\n")
print("="*80 + "\n")

answer = query_database(question, db)

print("\n" + "="*80)
print("\n답변:")
display(Markdown(answer))

질문: 2021년에서 2022년 사이 지역별 인구증감률과 실질 GRDP 증감률을 비교해줘.


[1] SQL 생성 중...
    WITH pop AS (
    SELECT
        "행정구역별(읍면동)" AS region,
        MAX(CASE WHEN "연령별" = '합계' THEN "2021" END) AS pop_2021,
        MAX(CASE WHEN "연령별" = '합계' THEN "2022" END) AS pop_2022
    FROM population
    GROUP BY "행정구역별(읍면동)"
),
grdp AS (
    SELECT
        "행정구역별(1)" AS region,
        MAX(CASE WHEN "행정구역별(2)" = '소계' THEN "2021_1"::NUMERIC END) AS grdp_2021_real,
        MAX(CASE WHEN "행정구역별(2)" = '소계' THEN "2022_1"::NUMERIC END) AS grdp_2022_real
    FROM "GRDP"
    WHERE "행정구역별(1)" <> '행정구역별(1)'
    GROUP BY "행정구역별(1)"
)
SELECT
    p.region AS "지역",
    p.pop_2021 AS "2021인구",
    p.pop_2022 AS "2022인구",
    ROUND((p.pop_2022 - p.pop_2021) * 100.0 / NULLIF(p.pop_2021, 0), 2) AS "인구증감률(%)",
    g.grdp_2021_real AS "2021실질GRDP",
    g.grdp_2022_real AS "2022실질GRDP",
    ROUND((g.grdp_2022_real - g.grdp_2021_real) * 100.0 / NULLIF(g.grdp_2021_real, 0), 2) AS "실질GRDP증감률(%)"
FROM pop p
LEFT JOIN grdp g
    ON 

2021년→2022년 **지역별 인구증감률**과 **실질 GRDP 증감률**을 비교하면, 공통으로 포함된 지역은 **14개 시도**입니다.  
- **공통연도:** 2021년, 2022년  
- **GRDP 기준:** **실질 GRDP**(2021_1, 2022_1 사용)  
- **포함 지역:** 강원특별자치도, 경기도, 경상남도, 경상북도, 대구광역시, 대전광역시, 부산광역시, 서울특별시, 울산광역시, 인천광역시, 전북특별자치도, 제주특별자치도, 충청남도, 충청북도

## 전체 비교 요약
- **인구 증가 지역은 4곳**, 나머지 **10곳은 인구 감소**였습니다.
- 반면 **실질 GRDP는 14개 지역 모두 증가**했습니다.
- 즉, **인구는 일부 지역에서 줄었지만 실질 GRDP는 모든 지역에서 늘어난** 모습입니다.

## 지역별 비교
| 지역 | 인구증감률(%) | 실질 GRDP증감률(%) |
|---|---:|---:|
| 강원특별자치도 | 0.40 | 1.30 |
| 경기도 | 0.48 | 3.24 |
| 경상남도 | -0.76 | 3.53 |
| 경상북도 | -0.57 | 0.63 |
| 대구광역시 | -0.88 | 2.17 |
| 대전광역시 | -0.41 | 1.84 |
| 부산광역시 | -0.86 | 1.73 |
| 서울특별시 | -0.58 | 2.64 |
| 울산광역시 | -0.91 | 3.79 |
| 인천광역시 | 1.08 | 6.84 |
| 전북특별자치도 | -0.72 | 1.31 |
| 제주특별자치도 | 0.49 | 4.56 |
| 충청남도 | 0.79 | 2.84 |
| 충청북도 | 0.01 | 4.18 |

## 눈에 띄는 지역
- **인구증가율 최고:** 인천광역시 **1.08%**
- **인구증가율 최저:** 울산광역시 **-0.91%**
- **실질 GRDP증가율 최고:** 인천광역시 **6.84%**
- **실질 GRDP증가율 최저:** 경상북도 **0.63%**

## 해석
- **인구가 증가한 지역**: 인천, 충남, 제주, 강원, 충북  
  - 이 중 인천은 **인구와 실질 GRDP가 모두 가장 크게 증가한 지역**입니다.
- **인구가 감소한 지역**도 대부분 실질 GRDP는 증가하여,  
  **인구 증감과 실질 GRDP 증감이 반드시 같은 방향으로 움직이지는 않음**을 보여줍니다.
- 특히 **경상남도(-0.76%)**, **울산광역시(-0.91%)**처럼 인구는 줄었지만 실질 GRDP는 각각 **3.53%**, **3.79%** 증가했습니다.

원하시면 제가 이 결과를 바탕으로  
1) **인구증감률과 GRDP증감률의 차이(격차)**, 또는  
2) **인구 증가 지역/감소 지역으로 나눠서 요약**해드릴게요.

## 10. 다양한 질문으로 테스트

조회, 정렬, JOIN, 집계, 1인당 지표와 증감률을 포함한 6개 질문으로 테스트합니다.

In [13]:
# TODO: 팀 데이터에 맞는 다양한 질문들을 작성하세요
# 기본 조회, JOIN, 집계, 정렬 등 다양한 유형의 질문 포함

questions = [
    "경기도와 충남의 인구차이 및 GRDP 차이"
]

for q in questions:
    print(f"\n{'='*80}")
    print(f"질문: {q}")
    print("="*80 + "\n")

    try:
        answer = query_database(q, db)
        print("\n답변:")
        display(Markdown(answer))
    except Exception as e:
        print(f"오류: {e}")


질문: 경기도와 충남의 인구차이 및 GRDP 차이

[1] SQL 생성 중...
    WITH pop AS (
    SELECT
        CASE
            WHEN "행정구역별(읍면동)" IN ('경기도', '충청남도', '충남') THEN
                CASE
                    WHEN "행정구역별(읍면동)" = '충남' THEN '충청남도'
                    ELSE "행정구역별(읍면동)"
                END
        END AS region,
        "2025" AS population
    FROM population
    WHERE "연령별" = '합계'
      AND "행정구역별(읍면동)" IN ('경기도', '충청남도', '충남')
),
grdp AS (
    SELECT
        CASE
            WHEN "행정구역별(1)" IN ('경기도', '충청남도', '충남') THEN
                CASE
                    WHEN "행정구역별(1)" = '충남' THEN '충청남도'
                    ELSE "행정구역별(1)"
                END
        END AS region,
        CAST(REPLACE("2022", ',', '') AS BIGINT) AS grdp
    FROM GRDP
    WHERE "행정구역별(2)" = '소계'
      AND "행정구역별(1)" IN ('경기도', '충청남도', '충남')
),
merged AS (
    SELECT
        p.region,
        p.population,
        g.grdp
    FROM pop p
    JOIN grdp g ON p.region = g.region
)
SELECT
    MAX(CASE WHEN region = '경기도' T

경기도와 충남의 **인구차이**와 **GRDP 차이**는 다음과 같습니다.

### 1) 인구
- **경기도 인구:** 13,967,870명  
- **충남(충청남도) 인구:** 2,249,201명  
- **인구차이:** **11,718,669명**

### 2) GRDP
- **경기도 GRDP:** 587,328,570  
- **충남(충청남도) GRDP:** 140,928,250  
- **GRDP 차이:** **446,400,320**

### 비교 기준
- **인구 기준 연도:** `2025`
- **GRDP 기준 연도:** `2022`
- **포함 지역:** `경기도`, `충청남도(충남)`

즉, SQL 결과 기준으로 **경기도가 충남보다 인구는 1,171만 8,669명 많고, GRDP는 446,400,320만큼 큽니다.**

## 프로젝트 점검 체크리스트

**완료한 항목을 확인하세요:**

- [ ] CSV 파일 준비 및 데이터 확인 완료
- [ ] CSV 데이터 탐색 및 통계 분석 완료
- [ ] Supabase 연결 완료
- [ ] CSV 데이터 업로드 완료
- [ ] 직접 작성한 SQL 쿼리 테스트 완료 (최소 3개)
- [ ] Text2SQL 함수 구현 및 프롬프트 수정 완료
- [ ] 자연어 질문으로 SQL 생성 테스트 완료
- [ ] 완전한 Text2SQL 시스템 (답변 생성) 테스트 완료
- [ ] 최소 5개 이상의 다양한 질문으로 테스트 완료

---

## 추가 개선 아이디어

1. **프롬프트 개선**: SQL 생성 정확도 향상을 위한 예시 추가
2. **에러 핸들링**: SQL 오류 발생 시 재시도 로직 구현
3. **쿼리 검증**: 생성된 SQL이 안전한지 검증하는 로직 추가
4. **결과 포맷팅**: 테이블 형태로 결과 출력
5. **쿼리 히스토리**: 실행한 쿼리와 결과를 저장하여 재사용
6. **고급 SQL**: 서브쿼리, CTE, 윈도우 함수 활용